# Exploratory Data Analysis — Store Sales (Corporación Favorita)

**Goal:** Understand sales patterns, seasonality, external factors (oil, holidays, promotions) — before any modelling.

**Dataset:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)  
54 stores · 33 product families · 4.5 years (2013–2017) · Ecuador's largest grocery chain

---
**Table of Contents**
1. Setup & Load
2. Dataset Overview
3. Sales Distribution by Store & Family
4. Time Series Patterns (seasonality, trend)
5. External Features (oil price, holidays, promotions)
6. Correlation Analysis
7. Key Insights for Modelling

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

In [ ]:
# Load raw Store Sales data
RAW = '../data/raw/'
train  = pd.read_csv(RAW + 'train.csv', parse_dates=['date'])
oil    = pd.read_csv(RAW + 'oil.csv',   parse_dates=['date'])
hol    = pd.read_csv(RAW + 'holidays_events.csv', parse_dates=['date'])
stores = pd.read_csv(RAW + 'stores.csv')
trans  = pd.read_csv(RAW + 'transactions.csv', parse_dates=['date'])

# Merge oil prices (fill weekends)
oil = oil.set_index('date').reindex(
    pd.date_range(oil.date.min(), oil.date.max(), freq='D')
).interpolate().reset_index().rename(columns={'index':'date','dcoilwtico':'oil_price'})

train = train.merge(oil[['date','oil_price']], on='date', how='left')

print(f'Train:  {len(train):,} rows | {train.date.min().date()} -> {train.date.max().date()}')
print(f'Stores: {train.store_nbr.nunique()} | Families: {train.family.nunique()}')
print(f'Series: {train.store_nbr.nunique() * train.family.nunique():,}')
train.head()

---
## 1. Dataset Overview

In [ ]:
print('=== SHAPE ==='); print(f'  {len(train):,} rows x {train.shape[1]} cols')
print('\n=== MISSING ==='); print(train.isnull().sum()[train.isnull().sum()>0])
print('\n=== SALES STATS ===')
print(train['sales'].describe().round(2))
print(f'\nZero-sales rows: {(train.sales==0).mean():.1%} (product not on shelf)')

In [ ]:
# Store metadata
print('Store types:', stores.type.value_counts().to_dict())
print('Store clusters:', stores.cluster.nunique())
print('Cities:', stores.city.nunique())
stores.head(8)

---
## 2. Sales by Store & Family

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Top stores by total sales
store_sales = train.groupby('store_nbr')['sales'].sum().sort_values(ascending=False).head(15)
axes[0].bar(store_sales.index.astype(str), store_sales.values/1e6, color='#3498db')
axes[0].set_title('Top 15 Stores by Total Sales', fontweight='bold')
axes[0].set_xlabel('Store #'); axes[0].set_ylabel('Total Sales (M units)')
axes[0].tick_params(axis='x', rotation=45)

# Top families by total sales
fam_sales = train.groupby('family')['sales'].sum().sort_values(ascending=False).head(15)
colors = ['#2ecc71' if i==0 else '#3498db' for i in range(len(fam_sales))]
axes[1].barh(fam_sales.index, fam_sales.values/1e6, color=colors)
axes[1].set_title('Top 15 Product Families by Total Sales', fontweight='bold')
axes[1].set_xlabel('Total Sales (M units)')

plt.tight_layout()
plt.show()

print('Top 5 families:', fam_sales.index[:5].tolist())

---
## 3. Time Series Patterns

In [ ]:
# Aggregate total daily sales across all stores
daily = train.groupby('date')['sales'].sum().reset_index()
daily['year']  = daily.date.dt.year
daily['month'] = daily.date.dt.month
daily['dow']   = daily.date.dt.dayofweek

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Full time series
axes[0].plot(daily.date, daily.sales/1e6, color='#2c3e50', lw=0.8, alpha=0.8)
axes[0].set_title('Total Daily Sales (All Stores, All Families)', fontweight='bold')
axes[0].set_ylabel('Units Sold (M)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Weekly seasonality
dow_avg = daily.groupby('dow')['sales'].mean()
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
axes[1].bar(days, dow_avg.values/1e6, color=['#e74c3c' if d=='Sat' or d=='Sun' else '#3498db' for d in days])
axes[1].set_title('Average Sales by Day of Week', fontweight='bold')
axes[1].set_ylabel('Avg Units (M)')

# Monthly seasonality
mon_avg = daily.groupby('month')['sales'].mean()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[2].bar(months, mon_avg.values/1e6, color='#9b59b6')
axes[2].set_title('Average Sales by Month', fontweight='bold')
axes[2].set_ylabel('Avg Units (M)')

fig.suptitle('Temporal Sales Patterns — Corporacion Favorita', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Weekend vs weekday lift: {dow_avg[[5,6]].mean()/dow_avg[:5].mean():.2f}x')

**Time series findings:**
- Clear **weekly seasonality**: weekends (especially Saturday) are highest
- **Upward trend** over 2013-2017
- **December spike** every year (holiday shopping)
- **2016 dip**: Ecuador earthquake (April 2016) caused temporary sales drop

---
## 4. External Features

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Oil price vs total sales
oil_daily = daily.merge(oil[['date','oil_price']], on='date', how='left')

ax2 = axes[0].twinx()
axes[0].plot(oil_daily.date, oil_daily.sales/1e6, color='#2c3e50', lw=0.8, alpha=0.7, label='Sales')
ax2.plot(oil_daily.date, oil_daily.oil_price, color='#e67e22', lw=1.2, alpha=0.9, label='Oil Price')
axes[0].set_ylabel('Total Sales (M units)', color='#2c3e50')
ax2.set_ylabel('Oil Price (USD)', color='#e67e22')
axes[0].set_title('Sales vs Oil Price — Ecuador is oil-dependent', fontweight='bold')
lines1, labels1 = axes[0].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[0].legend(lines1+lines2, labels1+labels2, loc='upper left')

# Promotion effect
daily_promo = train.groupby(['date','onpromotion'])['sales'].sum().reset_index()
promo_effect = daily_promo.groupby('onpromotion')['sales'].mean()
axes[1].bar(['Not on Promo', 'On Promotion'], promo_effect.values/1e3,
            color=['#95a5a6','#e74c3c'], edgecolor='white')
for i, v in enumerate(promo_effect.values):
    axes[1].text(i, v/1e3+5, f'{v/1e3:.0f}K', ha='center', fontweight='bold')
axes[1].set_title('Average Daily Sales: Promotion vs No Promotion', fontweight='bold')
axes[1].set_ylabel('Avg Daily Sales (K units)')
promo_lift = promo_effect.iloc[1] / promo_effect.iloc[0]
axes[1].text(0.5, 0.85, f'Promotion lift: {promo_lift:.1f}x',
             ha='center', transform=axes[1].transAxes, fontsize=12,
             bbox=dict(boxstyle='round', facecolor='#ffeaa7'))

plt.tight_layout()
plt.show()

print(f'Promotion sales lift: {promo_lift:.2f}x ({(promo_lift-1)*100:.0f}% more units sold)')

In [ ]:
# Holiday effect
national_hols = hol[(hol.type.isin(['Holiday','Transfer'])) & 
                    (hol.locale=='National') & (~hol.transferred)]
holiday_dates = set(national_hols.date)

daily['is_holiday'] = daily.date.isin(holiday_dates).astype(int)
hol_effect = daily.groupby('is_holiday')['sales'].mean()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['Normal Day', 'National Holiday'], hol_effect.values/1e6,
               color=['#3498db','#e74c3c'], edgecolor='white')
for bar, val in zip(bars, hol_effect.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'{val/1e6:.2f}M', ha='center', fontweight='bold')
ax.set_title('Average Daily Sales: Normal vs Holiday', fontweight='bold')
ax.set_ylabel('Avg Units (M)')
hol_lift = hol_effect.iloc[1]/hol_effect.iloc[0]
ax.text(0.5, 0.85, f'Holiday effect: {hol_lift:.2f}x',
        ha='center', transform=ax.transAxes, fontsize=12,
        bbox=dict(boxstyle='round', facecolor='#ffeaa7'))
plt.tight_layout()
plt.show()

print(f'National holidays: {len(national_hols)} events in dataset')
print(f'Holiday lift: {hol_lift:.2f}x')

---
## 5. Oil Price Analysis

In [ ]:
# Ecuador is oil-dependent — oil price shocks affect consumer spending
# 2014: oil crash from ~$100 to ~$50 → economic stress → austerity measures

oil_merged = daily.merge(oil[['date','oil_price']], on='date', how='left').dropna()
corr = oil_merged[['sales','oil_price']].corr().iloc[0,1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: oil price vs sales
axes[0].scatter(oil_merged.oil_price, oil_merged.sales/1e6, 
                alpha=0.3, s=5, color='#e67e22')
z = np.polyfit(oil_merged.oil_price.dropna(), 
               oil_merged.sales[oil_merged.oil_price.notna()]/1e6, 1)
p = np.poly1d(z)
x_line = np.linspace(oil_merged.oil_price.min(), oil_merged.oil_price.max(), 100)
axes[0].plot(x_line, p(x_line), 'r-', lw=2)
axes[0].set_xlabel('Oil Price (USD)')
axes[0].set_ylabel('Daily Sales (M units)')
axes[0].set_title(f'Oil Price vs Sales (r={corr:.3f})', fontweight='bold')

# Oil price over time with annotated events
axes[1].plot(oil.date, oil.oil_price, color='#e67e22', lw=1.5)
axes[1].axvline(pd.Timestamp('2014-06-01'), color='red', ls='--', alpha=0.7)
axes[1].text(pd.Timestamp('2014-06-01'), 105, 'Oil crash\n2014', fontsize=9, color='red')
axes[1].axvline(pd.Timestamp('2016-04-16'), color='purple', ls='--', alpha=0.7)
axes[1].text(pd.Timestamp('2016-04-16'), 25, 'Earthquake\nApr 2016', fontsize=9, color='purple')
axes[1].set_title('Oil Price Timeline (Ecuador)', fontweight='bold')
axes[1].set_ylabel('USD/barrel')

plt.tight_layout()
plt.show()

print(f'Oil-Sales correlation: {corr:.3f}')
print('Interpretation: positive correlation — when oil prices drop, Ecuador suffers')
print('economically and consumer spending falls. A key external regressor.')

---
## 6. Key Insights for Modelling

In [ ]:
# Heatmap: avg sales by store_type and family (top 10 families)
merged = train.merge(stores[['store_nbr','type']], on='store_nbr')
top_fams = train.groupby('family')['sales'].sum().nlargest(10).index
pivot = merged[merged.family.isin(top_fams)].groupby(
    ['type','family'])['sales'].mean().unstack().fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot/1e3, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Avg Daily Sales (K)'})
ax.set_title('Average Daily Sales by Store Type x Product Family (top 10)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('=' * 60)
print('KEY INSIGHTS FOR MODELLING')
print('=' * 60)
print()
print('1. WEEKLY SEASONALITY is the strongest pattern.')
print('   Saturday sales are highest — must use lag_7 as primary feature.')
print()
print('2. OIL PRICE is a meaningful external regressor (r > 0).')
print('   Ecuador is oil-dependent — oil price shocks reduce consumer spending.')
print()
print('3. PROMOTIONS lift sales significantly.')
print('   onpromotion flag is a critical feature in every model.')
print()
print('4. HOLIDAYS have measurable effects.')
print('   National holidays increase aggregate demand.')
print()
print('5. ANNUAL SEASONALITY matters.')
print('   December peak every year -> lag_364 (same day last year) is critical.')
print()
print('6. EXTERNAL SHOCKS are unpredictable but important.')
print('   Ecuador earthquake (Apr 2016) caused a demand drop not capturable by lags.')
print('   Foundation model (Chronos-2) handles these better than pure lag models.')
print()
print('7. HIGH ZERO-SALES RATE (~{:.0f}%)'.format((train.sales==0).mean()*100))
print('   Products not carried in every store. RMSLE handles this via log transform.')